In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from modules.features import HolisticLandmarksFeatureExtracter, FeatureTransformer
from modules.image_annotators import RealLifeTestAnnotator
from modules.model_factory import (SimpleModel,
                                  SimpleModelWithNoise,
                                  SimpleLSTMModel)
from modules.features.transformations import (FeatureWeights, 
                                              FeatureCentering,
                                              FeatureNormalizer,
                                              FeatureRemoveLegData)

from modules.utilities.file import load_dataset
from modules.process import train_and_validate_model

In [2]:
DATA_PATH: Path = PROJECT_ROOT / 'mp_data'
VALIDATION_DATA_PATH: Path = PROJECT_ROOT / 'mp_data_new_test'
TENSORFLOW_MODEL_PATH: Path = Path('./output_models/simple_model.keras')
NR_OF_EPOCHS: int = 50

feature_extracter = HolisticLandmarksFeatureExtracter()

#feature_transformer = FeatureTransformer()
#feature_transformer = FeatureTransformer([FeatureCentering(), FeatureWeights(10.0, 10.0, 2.0, 0.5)])
feature_transformer = FeatureTransformer([FeatureRemoveLegData(), FeatureCentering(), FeatureWeights(10.0, 10.0, 2.0, 0.5)])


In [3]:
baseline_model, baseline_training = train_and_validate_model(model_factory=SimpleModel(),
                         feature_extracter=feature_extracter,
                         feature_transformer=FeatureTransformer(),
                         train_dataset_path=DATA_PATH,
                         validation_dataset_path=VALIDATION_DATA_PATH,
                         nr_of_epochs=NR_OF_EPOCHS)

normalized_model, normalized_training = train_and_validate_model(model_factory=SimpleModel(),
                         feature_extracter=feature_extracter,
                         feature_transformer=feature_transformer,
                         train_dataset_path=DATA_PATH,
                         validation_dataset_path=VALIDATION_DATA_PATH,
                         nr_of_epochs=NR_OF_EPOCHS)
   

Actions: ['idle', 'poes', 'slapen', 'vogel']
Number of sequences: 120
Number of classes: 4
Input shape: (120, 30, 1659)
Input dtype: float32
NaN values: 0
Infinite values: 0
idle: 30 sequences
poes: 30 sequences
slapen: 30 sequences
vogel: 30 sequences
Epoch 1/50


ValueError: Argument `output` must have rank (ndim) `target.ndim - 1`. Received: target.shape=(None, 4), output.shape=(None, 4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, history in [("Baseline", baseline_training), ("Normalized", normalized_training)]:
    axes[0].plot(history.history["loss"], label=label)
    axes[1].plot(history.history["categorical_accuracy"], label=label)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Training accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.05))
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Adding noise layer to the model to improve generalization and reduce overfitting
train_and_validate_model(model_factory=SimpleModelWithNoise(),
                         feature_extracter=feature_extracter,
                         feature_transformer=feature_transformer,
                         train_dataset_path=DATA_PATH,
                         validation_dataset_path=VALIDATION_DATA_PATH,
                         nr_of_epochs=NR_OF_EPOCHS)

In [ ]:
train_and_validate_model(model_factory=SimpleLSTMModel(),
                         feature_extracter=feature_extracter,
                         feature_transformer=feature_transformer,
                         train_dataset_path=DATA_PATH,
                         validation_dataset_path=VALIDATION_DATA_PATH,
                         nr_of_epochs=NR_OF_EPOCHS)